# 第3周 · Python III：Pandas、调试、类与对象、第一个分类器

**本周结束时你能做到**：用 Pandas 读取、筛选、分组真实表格并处理缺失值；按固定流程排查报错；读懂并写出一个简单的类；用 scikit-learn 训练你的第一个分类器。

本周的数据是经典的 Titanic 乘客名单：891 名乘客，谁活了下来？这是一份真实的历史数据，也是无数人做数据分析的第一份表格。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from zero2researcher import data
from zero2researcher.checks import week03 as ck

---
## S1 · Pandas：会说话的表格

NumPy 数组里只有数字；真实数据往往是**混合类型的表格**：名字是字符串、年龄是数字、有些格子是空的。Pandas 的 `DataFrame` 就是为这种表格而生的。

读表格用 `pd.read_csv`。课程工具包里存了一份 Titanic 数据的副本，`data.load_titanic()` 内部就是下面这一行（之所以不用 `seaborn.load_dataset`，是因为它需要联网下载——在 Colab 上每开一个新会话都要重新下载一次）：

In [ ]:
titanic = pd.read_csv(data.path("titanic.csv"))   # 与 data.load_titanic() 完全等价
print(titanic.shape)
titanic.head()

891 行 × 15 列：每一行是一名乘客，每一列是一个属性。最关心的列是 `survived`（0 = 遇难，1 = 生还）。

### 列选择

In [ ]:
print(titanic["age"].head())                 # 一列 → Series
titanic[["sex", "age", "survived"]].head()   # 列名的列表 → 还是 DataFrame

一列是 `Series`（带标签的一维数组），它有和 NumPy 数组一样的统计方法：

In [ ]:
print(titanic["age"].mean(), titanic["age"].median())
print(titanic["survived"].mean())            # 0/1 的均值就是生还比例

### 条件筛选：统一用 `.loc[行条件, 列]`

Pandas 有好几种筛选写法，混用容易踩坑。本课程统一用 `.loc`——筛选用它，改值也用它（下面马上会看到为什么）：

In [ ]:
children = titanic.loc[titanic["age"] < 10, ["sex", "age", "survived"]]
print(len(children), "名 10 岁以下的儿童")
children.head()

多个条件用 `&`（且）、`|`（或）连接，每个条件都要加括号——和第 2 周的布尔掩码一模一样：

In [ ]:
titanic.loc[(titanic["sex"] == "female") & (titanic["pclass"] == 1)].head(3)

**改值也用 `.loc`。** 新手最常写的改值方式是“链式赋值”：`df[条件]["列"] = 值`——先筛出一张子表，再改子表的那一列。这一行**改不到原表**：

- pandas 2.x 会弹出一条 `SettingWithCopyWarning`（“你可能在改一个副本”）；
- pandas 3.0 起默认“写时复制”（Copy-on-Write）：筛出来的子表就是一份独立的副本，pandas 会给出 `ChainedAssignmentError` 警告，原表纹丝不动。

两种版本下，正确写法都是一步到位的 `df.loc[行条件, "列"] = 值`。在一份副本上试一次（下一格**会**弹出一条警告，这是故意的）：

In [ ]:
demo = titanic.copy()                                # 在副本上演示，不动 titanic 本身
demo[demo["age"] < 1]["age"] = 1                     # ❌ 链式赋值：弹出警告，而且没有改到 demo
print("链式赋值之后，不满 1 岁的乘客还有", int((demo["age"] < 1).sum()), "名")

demo.loc[demo["age"] < 1, "age"] = 1                  # ✅ .loc：行条件和列名写在同一对方括号里
print(".loc 赋值之后，不满 1 岁的乘客还有", int((demo["age"] < 1).sum()), "名")

### 缺失值

真实数据几乎总有空格子（`NaN`）。`isna()` 把每个格子变成“是否缺失”，再按列求和：

In [ ]:
titanic.isna().sum()

`age` 缺 177 个、`deck` 缺 688 个（`embarked` 和 `embark_town` 缺的是同样的 2 行）。处理缺失值没有万能答案，要看缺了多少、为什么缺：

- `deck`（甲板层，A–G）缺了 77%——这一列基本没法用；
- `age` 缺 20%——直接丢掉这 177 名乘客太可惜，常用做法是**用中位数填充**。

用 `fillna` 写出填充后的年龄列 `age_filled`：

In [ ]:
# ✍️ 跟着老师一起写
pass
print(int(age_filled.isna().sum()), "个缺失（填充后）")
ck.age_filled(age_filled)

### groupby：分组统计，一行回答一个研究问题

“女性的生还率比男性高多少？”——`groupby` 按某一列分组，再对每组做统计：

In [ ]:
# ✍️ 跟着老师一起写
pass
print(surv_by_sex)
ck.survival_by_sex(surv_by_sex)

女性 74.2%、男性 18.9%——“妇女优先”不是传说。再按舱位分组：

In [ ]:
# ✍️ 跟着老师一起写
pass
print(surv_by_class)
ck.survival_by_class(surv_by_class)

一等舱 63%、三等舱 24%。还可以同时按两列分组：

In [ ]:
titanic.groupby(["sex", "pclass"])["survived"].mean()

一等舱女性 96.8%，三等舱男性 13.5%。两行代码、一个清晰的结论——这就是 Pandas 的威力。

### merge：合并两张表

真实项目的数据常常分散在多张表里，`pd.merge` 按共同的列把它们拼起来（和 Excel 的 VLOOKUP 一个意思）：

In [ ]:
passengers = pd.DataFrame({"passenger_id": [1, 2, 3, 4],
                           "name": ["Allen", "Bowen", "Chen", "Du"]})
tickets = pd.DataFrame({"passenger_id": [1, 2, 4],
                        "ticket_price": [71.3, 7.9, 53.1]})
pd.merge(passengers, tickets, on="passenger_id", how="left")

`how="left"` 保留左表的所有行；Chen 没有票价记录，拼出来就是 `NaN`——merge 也是缺失值的常见来源。

---
## S2 · 调试：出错是常态，排查有流程

写代码的真相：**大部分时间花在找错上**，新手和老手的区别只是找得快不快。三板斧：

1. **读 traceback 的最后一行**：错误类型 + 一句人话，然后从下往上找到自己写的那行。
2. **print 调试**：在可疑的位置打印中间变量的值、`type()` 或 `.shape`，看它和你想象的是否一致。
3. **最小可复现例子**：把出错的代码剥到只剩几行还能复现——一半情况下，剥着剥着自己就明白了。

先故意制造一个错误（这个单元格**应该**报错）：

In [ ]:
titanic["Age"].mean()    # 列名其实是小写的 age

最后一行 `KeyError: 'Age'`——“没有叫 Age 的键”。列名记不清时，先 `print(titanic.columns)`。

（Jupyter 还有 `%debug` 魔法命令：出错后在新单元格运行它，可以“钻进”出错现场逐层查看变量。今天不演示，记得有这个工具即可。）

print 调试示范：下面的函数第一次调用算出了 `nan`，在函数里加一个 print 立刻看出原因：

In [ ]:
def survival_rate(df, column, value):
    subset = df.loc[df[column] == value]
    print("子集行数:", len(subset))        # print 调试：检查中间结果
    return subset["survived"].mean()

print(survival_rate(titanic, "pclass", "1"))   # nan！行数是 0——"1" 是字符串，这一列存的是数字
print(survival_rate(titanic, "pclass", 1))

### 类与对象：把数据和操作打包在一起

做实验你一定写过**实验方案**（protocol）：规定了有哪些材料、能做哪些操作。每次按方案做一次实验，得到一次具体的**实验运行**，各有各的数据。

类（class）与对象（object）正是这个关系：

- **类 = 实验方案**：规定这类东西有哪些数据（属性）、能做哪些操作（方法）；
- **对象 = 一次具体的实验运行**：按方案创建出来的实例，数据互不干扰。

经典入门例子：银行账户。

In [ ]:
class BankAccount:
    """一个最简单的银行账户：存款、取款、查询余额"""

    def __init__(self, owner, balance=0):
        """创建账户时自动运行：记下户主和初始余额"""
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        """存款"""
        # ✍️ 跟着老师一起写
        pass

    def withdraw(self, amount):
        """取款：余额不足时不扣款，打印提示"""
        # ✍️ 跟着老师一起写
        pass

    def get_balance(self):
        """查询余额"""
        # ✍️ 跟着老师一起写
        pass

ck.bank_account(BankAccount)

In [ ]:
acct = BankAccount("Ada", 100)     # 按方案创建一个对象
acct.deposit(50)
acct.withdraw(30)
print(acct.owner, acct.get_balance())

acct2 = BankAccount("Bob")         # 同一个类的另一个对象，数据互不干扰
print(acct2.owner, acct2.get_balance())

读懂这段代码需要认得三个记号：

- `__init__` 在创建对象时自动运行；
- `self` 指“这个对象自己”——`self.balance` 是属性（数据），各对象一份；
- `acct.deposit(50)` 是方法调用（操作）。

**为什么现在学这个**：第 8 周起你写的每一个神经网络都是一个类：

```python
class MyModel(nn.Module):      # 我的模型是 nn.Module 的一种
    def __init__(self):
        ...                    # 定义有哪些层
    def forward(self, x):
        ...                    # 定义数据怎么流过这些层
```

今天不需要懂 `nn.Module`，只要求你**认得这个模式**：`class`、`__init__`、`self`、方法。

---
## S3 · Git/GitHub 与第一个分类器

### 代码也需要实验记录本：版本控制

Git 是代码的版本控制系统：每次**提交（commit）**是一次快照，记下“谁、何时、改了什么、为什么”。GitHub 是存放和分享这些快照的网站。对照你熟悉的场景：

| 实验室 | 编程 |
| --- | --- |
| 实验记录本（每页有日期和签名） | 提交历史（每次 commit 有作者和说明） |
| “最终版_v3_真的最终版.docx” | 同一个文件 + 完整的历史版本 |
| 共享的 protocol 文件夹 | GitHub 仓库 |

本课程你暂时只需要一个操作：在 Colab 里 **File → Save a copy in GitHub**，把每周的 notebook 存进自己的仓库，commit message 写一句本周做了什么。课程资料本身就放在 GitHub 上——你一直在用它。

### 第一个分类器

接下来十分钟，你会训练出自己的第一个机器学习模型。**今天只看“怎么用”，不问“为什么”**——后面十七周都在回答为什么。

Iris（鸢尾花）是最经典的入门数据集：150 朵花 × 4 个测量值（花萼与花瓣的长宽），3 个品种。任务：由测量值判断品种。

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

X, y = load_iris(return_X_y=True)
print(X.shape, y.shape)            # (150, 4)：150 个样本、每个 4 个特征——第2周的老朋友

# ✍️ 跟着老师一起写
pass
print(f"准确率：{iris_acc:.4f}")
print("前5朵的预测:", model.predict(X[:5]), " 真实品种:", y[:5])
ck.iris_accuracy(iris_acc)

三个动词就是全部咒语：`fit`（训练）、`predict`（预测）、`score`（打分）。97% 的花被判对了。

### 换上 Titanic：预测谁能生还

同一套 `fit / predict / score`，换真实数据。用四个特征：性别、舱位、年龄（缺失用中位数填充）、票价。模型只吃数字，所以先把 `sex` 变成 0/1：

In [ ]:
from sklearn.model_selection import train_test_split

features = pd.DataFrame({
    "sex": (titanic["sex"] == "female").astype(int),            # female=1, male=0
    "pclass": titanic["pclass"],
    "age": titanic["age"].fillna(titanic["age"].median()),  # 中位数用了全部乘客（含测试集），第5周会讲为什么不够严谨
    "fare": titanic["fare"],
})
target = titanic["survived"]

# 留出 1/4 的乘客不参加训练，专门用来打分（为什么要这样？第5周详谈）
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42, stratify=target)

# ✍️ 跟着老师一起写
pass
print(f"在没见过的乘客上的准确率：{titanic_acc:.4f}")
ck.titanic_accuracy(titanic_acc)

约 76%。注意：瞎猜“全部遇难”也能对 62%（生还率只有 38%），所以模型确实学到了东西，但远非完美。能不能更好？用什么特征？怎么公平地比较两个模型？——这正是第 4–6 周的内容。

---
## S4 · 从湿实验到 AI 研究：你已有的能力都能迁移

学了三周编程，退后一步看全局：**你在实验台边练出的科研直觉，几乎每一条都在 AI 研究中有对应物。**

| 实验生物学 | AI Research | 在本课程中出现 |
| --- | --- | --- |
| 提出科学假设 | 提出模型或算法假设 | 第8周 |
| 实验组与对照组 | 新模型与基线 | 第6周基线阶梯 |
| 改变实验条件 | 修改架构、损失或超参数 | 第9、10周 |
| 控制变量 | 一次只改一个超参数 | 第9周 |
| 生物学重复 | 多随机种子、交叉验证折 | 第5周 |
| 机制验证 | 消融实验 | 第10、12周 |
| 污染与假阳性 | 数据泄漏 | 第5、6周 |
| 标准品与阳性对照 | 合成数据与已知答案的 sanity check | 第4、7周 |
| 读文献、重复他人实验 | 论文阅读与复现 | 第13–15周 |
| 组会汇报与审稿 | 研究交流与同行评审 | 第12、20周 |

把一个你熟悉的实验逐条改写成计算实验：

> **湿实验**：“药物 X 能否提高细胞存活率？”——处理组加药，对照组不加，其余条件完全一致，3 个生物学重复，比较均值。
>
> **计算实验**：“特征 X 能否提高预测准确率？”——实验组用特征 X 训练，对照组不用，其余设定完全一致，3 个随机种子，比较平均准确率。

同一个思维模板。课程后半段写实验记录、设计消融、跑多种子，用的全是你熟悉的套路，只是研究对象从细胞换成了模型。

**AI 论文的标准结构**也和生物论文同构：Abstract → Introduction（问题与假设）→ Related Work（文献综述）→ Methods（模型与实验设置）→ Experiments（结果与消融）→ Conclusion。第 13 周开始读论文时，你不会陌生。

### 本周小结

- Pandas：`read_csv`、列选择、`.loc` 条件筛选、`isna`/`fillna`、`groupby`、`merge`
- 调试三板斧：读 traceback 最后一行、print 中间变量、最小可复现例子
- 类与对象：类 = 实验方案，对象 = 一次实验运行；`__init__`、`self`、方法
- 第一个分类器：`fit` / `predict` / `score`
- 你的实验科研思维可以整体迁移到 AI 研究

接下来是 **Python 阶段检查**（`03_checkpoint.ipynb`）：45 分钟，一份新数据，独立完成读取 → 清洗 → 汇总 → 作图。